# Exploratory Data Analysis (EDA)
Brain Tumor MRI Dataset

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

data_dir = Path('../data/raw')
classes = ['glioma', 'meningioma', 'notumor', 'pituitary']

## 1. Class Distribution

In [ ]:
class_counts = Counter()
for split in ['Training', 'Testing']:
    split_dir = data_dir / split
    if split_dir.exists():
        for cls in classes:
            cls_dir = split_dir / cls
            if cls_dir.exists():
                class_counts[cls] += len(list(cls_dir.glob('*.jpg')))

plt.figure(figsize=(10, 6))
sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()))
plt.title('Total Number of Images per Class')
plt.ylabel('Count')
plt.show()

total = sum(class_counts.values())
print("Class Imbalance Ratio:")
for cls, count in class_counts.items():
    print(f"{cls}: {count/total:.2%}")

## 2. Sample Images Grid

In [ ]:
import random
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, cls in enumerate(classes):
    cls_dir = data_dir / 'Training' / cls
    if cls_dir.exists():
        images = list(cls_dir.glob('*.jpg'))
        samples = random.sample(images, min(4, len(images)))
        for j, img_path in enumerate(samples):
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].set_title(f"{cls} ({img.shape})")
            axes[i, j].axis('off')
plt.tight_layout()
plt.show()

## 3. Image Size Distribution and Corrupt Image Check

In [ ]:
heights, widths = [], []
corrupt = []
for split in ['Training', 'Testing']:
    split_dir = data_dir / split
    if split_dir.exists():
        for cls in classes:
            cls_dir = split_dir / cls
            if cls_dir.exists():
                for img_path in cls_dir.glob('*.jpg'):
                    img = cv2.imread(str(img_path))
                    if img is None:
                        corrupt.append(img_path)
                    else:
                        heights.append(img.shape[0])
                        widths.append(img.shape[1])

print(f"Found {len(corrupt)} corrupt images.")
plt.figure(figsize=(10, 6))
sns.histplot(heights, label='Height', alpha=0.5)
sns.histplot(widths, label='Width', alpha=0.5)
plt.title('Image Size Distribution')
plt.legend()
plt.show()

## 4. Pixel Intensity Distribution

In [ ]:
plt.figure(figsize=(12, 8))
for cls in classes:
    cls_dir = data_dir / 'Training' / cls
    if cls_dir.exists():
        images = list(cls_dir.glob('*.jpg'))
        samples = random.sample(images, min(50, len(images))) # sample for speed
        pixels = []
        for img_path in samples:
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                pixels.extend(img.flatten())
        sns.kdeplot(pixels, label=cls)
plt.title('Pixel Intensity Distribution by Class (Sampled)')
plt.legend()
plt.show()